# Seminar 11: Tokenization, Encoder vs Decoder, and Hugging Face Workflow

Today is a practical Hugging Face lab.

Goals:
- inspect subword tokenization instead of guessing it,
- understand `input_ids`, `attention_mask`, and special tokens,
- use DistilBERT for masked-token prediction,
- use DistilGPT2 for next-token prediction and generation,
- summarize the encoder vs decoder difference from evidence you produced.

Expected rhythm: 5 medium board tasks, about 90 minutes total.


## 0. Setup

If you run this in Colab and `transformers` is missing, uncomment the install line first.


In [1]:
# If needed in Colab, uncomment:
# %pip install transformers sentencepiece accelerate -q

import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

pd.set_option('display.max_colwidth', 120)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cuda


### Provided Model Loading

This is not an exercise. Loading models is mechanical, and the interesting work starts after the tokenizer/model objects exist.

Important ideas:
- tokenizer and model must come from the same model family,
- DistilBERT is an encoder-style masked language model,
- DistilGPT2 is a decoder-style causal language model,
- GPT2 has no padding token by default, so we reuse the EOS token for batching.


In [2]:
MASKED_MODEL_NAME = 'distilbert-base-uncased'
CAUSAL_MODEL_NAME = 'distilgpt2'

bert_tok = AutoTokenizer.from_pretrained(MASKED_MODEL_NAME)
gpt_tok = AutoTokenizer.from_pretrained(CAUSAL_MODEL_NAME)

bert_mlm = AutoModelForMaskedLM.from_pretrained(MASKED_MODEL_NAME).to(device)
gpt_lm = AutoModelForCausalLM.from_pretrained(CAUSAL_MODEL_NAME).to(device)

bert_mlm.eval()
gpt_lm.eval()

gpt_tok.pad_token = gpt_tok.eos_token

print('Loaded:', MASKED_MODEL_NAME)
print('Loaded:', CAUSAL_MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded: distilbert-base-uncased
Loaded: distilgpt2


## 1. Exercise 1: Tokenizer Detective

Subword tokenization is easier to understand by inspecting examples.

Task:
- implement `compare_tokenizers(text)`,
- compare BERT-style and GPT-style tokenization for the same text,
- return a table with token strings, token ids, and token counts.

Function contract:
- `compare_tokenizers(text)` receives one Python string,
- it returns a `pandas.DataFrame`,
- required columns: `tokenizer`, `text`, `tokens`, `token_ids`, `num_tokens`.

Useful tokenizer methods:
- `tokenizer.tokenize(text)` returns token strings,
- `tokenizer.convert_tokens_to_ids(tokens)` converts token strings to integer ids.


In [4]:
tokens = bert_tok.tokenize('Hello world')
bert_tok.convert_tokens_to_ids(tokens)

[7592, 2088]

In [9]:
comparison_texts = [
    'Transformers changed natural language processing.',
    'unbelievable',
    'ChatGPT-like systems are everywhere now!',
    'bioinformatics-friendly tokenization?',
    'The     spacing is strange here.',
]



def compare_tokenizers(text):
    rows = []
    bert_tokens = bert_tok.tokenize(text)
    gpt_tokens = gpt_tok.tokenize(text)

    bert_token_ids = bert_tok.convert_tokens_to_ids(bert_tokens)
    gpt_token_ids = gpt_tok.convert_tokens_to_ids(gpt_tokens)

    # Add one row for BERT.
    # Add one row for GPT.
    # Each row should store tokens, token ids, and number of tokens.

    rows = [{'tokens': bert_tokens, 'token_ids': bert_token_ids, 'num_tokens': len(bert_tokens)}, {'tokens': gpt_tokens, 'token_ids': gpt_token_ids, 'num_tokens': len(gpt_tokens)}]

    return pd.DataFrame(rows)


tokenizer_comparison = compare_tokenizers(comparison_texts[2])
tokenizer_comparison


,tokens,token_ids,num_tokens
0,"[chat, ##gp, ##t, -, like, systems, are, everywhere, now, !]","[11834, 21600, 2102, 1011, 2066, 3001, 2024, 7249, 2085, 999]",10
1,"[Chat, G, PT, -, like, Ġsystems, Ġare, Ġeverywhere, Ġnow, !]","[30820, 38, 11571, 12, 2339, 3341, 389, 8347, 783, 0]",10


### Checks (Exercise 1)

In [ ]:
assert isinstance(tokenizer_comparison, pd.DataFrame)

required_columns = ['tokenizer', 'text', 'tokens', 'token_ids', 'num_tokens']
for column in required_columns:
    assert column in tokenizer_comparison.columns

assert len(tokenizer_comparison) == 2
assert 'DistilBERT' in tokenizer_comparison['tokenizer'].values
assert 'DistilGPT2' in tokenizer_comparison['tokenizer'].values

for row_index in range(len(tokenizer_comparison)):
    tokens = tokenizer_comparison.loc[row_index, 'tokens']
    token_ids = tokenizer_comparison.loc[row_index, 'token_ids']
    num_tokens = tokenizer_comparison.loc[row_index, 'num_tokens']
    assert len(tokens) == len(token_ids)
    assert num_tokens == len(tokens)
    assert num_tokens > 0

print('Exercise 1 passed.')


## 2. Exercise 2: From Text to Model Inputs

Token strings are only the beginning. Models receive tensors.

Task:
- tokenize a small batch with padding,
- inspect `input_ids`, `attention_mask`, and tokens position by position,
- compare how BERT and GPT inputs look.

Function contract:
- `show_model_inputs(tokenizer, texts)` receives a tokenizer and a list of strings,
- it returns a table with columns: `text_index`, `position`, `token`, `token_id`, `attention_mask`,
- it should use padding, because texts have different lengths.

Useful tokenizer call:
- `tokenizer(texts, return_tensors='pt', padding=True)` returns a dictionary with `input_ids` and `attention_mask`.

Useful tokenizer method:
- `tokenizer.convert_ids_to_tokens(ids)` maps ids back to token strings.


In [14]:
import pprint
pprint.pprint(bert_tok(["1234 skakd", '234', 'wrfe'], return_tensors='pt', padding=True))


{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0]]),
 'input_ids': tensor([[  101, 13138,  2549, 24053,  2243,  2094,   102],
        [  101, 22018,   102,     0,     0,     0,     0],
        [  101, 23277,  7959,   102,     0,     0,     0]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]])}


In [11]:
bert_tok.tokenize("1234 skakd",  padding=True)

['123', '##4', 'ska', '##k', '##d']

In [18]:
batch_texts = [
    'Deep learning is useful.',
    'Tokenizers turn text into numbers.',
]


def show_model_inputs(tokenizer, texts):
    rows = []

    # Tokenize texts with padding.
    tok_with_pd = tokenizer(texts, return_tensors='pt', padding=True)
    # Loop over each text and each token position.
    for i in range(len(batch_texts)):
      tokens = tokenizer.convert_ids_to_tokens(tok_with_pd['input_ids'][i])
      for j in range(len(tok_with_pd['input_ids'][i])):
        print(tokens[j])
        rows.append({
            'text_index': i,
            'position': j,
            'token': tokens[j],
            'token_id': tok_with_pd['input_ids'][i][j],
            'attention_mask': tok_with_pd['attention_mask'][i][j]
        })

    # Store token, token id, and attention mask value in rows.

    return pd.DataFrame(rows)


bert_input_table = show_model_inputs(bert_tok, batch_texts)
gpt_input_table = show_model_inputs(gpt_tok, batch_texts)

bert_input_table


[CLS]
deep
learning
is
useful
.
[SEP]
[PAD]
[PAD]
[PAD]
[CLS]
token
##izer
##s
turn
text
into
numbers
.
[SEP]
Deep
Ġlearning
Ġis
Ġuseful
.
<|endoftext|>
<|endoftext|>
Token
izers
Ġturn
Ġtext
Ġinto
Ġnumbers
.


,text_index,position,token,token_id,attention_mask
0,0,0,[CLS],tensor(101),tensor(1)
1,0,1,deep,tensor(2784),tensor(1)
2,0,2,learning,tensor(4083),tensor(1)
3,0,3,is,tensor(2003),tensor(1)
4,0,4,useful,tensor(6179),tensor(1)
5,0,5,.,tensor(1012),tensor(1)
6,0,6,[SEP],tensor(102),tensor(1)
7,0,7,[PAD],tensor(0),tensor(0)
8,0,8,[PAD],tensor(0),tensor(0)
9,0,9,[PAD],tensor(0),tensor(0)


### Checks (Exercise 2)

In [20]:
for table in [bert_input_table, gpt_input_table]:
    assert isinstance(table, pd.DataFrame)
    required_columns = ['text_index', 'position', 'token', 'token_id', 'attention_mask']
    for column in required_columns:
        assert column in table.columns
    assert len(table) > 0
    assert table['text_index'].nunique() == len(batch_texts)

print('Exercise 2 passed.')


Exercise 2 passed.


## 3. Exercise 3: Masked Language Modeling With BERT

BERT-style masked language models predict a missing token using both left and right context.

Task:
- implement `predict_masked_token(text, top_k=5)`,
- find the `[MASK]` position,
- run DistilBERT,
- return top predictions with probabilities.

Function contract:
- `text` must contain exactly one mask token: `bert_tok.mask_token`,
- output columns: `rank`, `token`, `token_id`, `probability`,
- probabilities should be sorted from largest to smallest.

Useful functions:
- `bert_tok(text, return_tensors='pt')` tokenizes one string,
- `bert_mlm(**inputs).logits` returns logits of shape `[batch, sequence_length, vocab_size]`,
- `F.softmax(logits, dim=-1)` converts logits to probabilities,
- `torch.topk(probabilities, k=top_k)` returns top values and ids.


In [29]:
masked_examples = [
    'Paris is the [MASK] of France.',
    'The programmer fixed the [MASK].',
    'The bank is near the [MASK].',
]


def predict_masked_token(text, top_k=5):
    inputs = bert_tok(text, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = bert_mlm(**inputs)

    input_ids = inputs['input_ids'][0]
    mask_position = None
    for position in range(len(input_ids)):
        if int(input_ids[position]) == bert_tok.mask_token_id:
            mask_position = position

    mask_logits = outputs.logits[0, mask_position]
    probabilities = F.softmax(mask_logits, dim=-1)
    top_probabilities, top_token_ids = torch.topk(probabilities, k=top_k)

    rows = []
    for i in range(top_k):
        token_id = int(top_token_ids[i])
        token = bert_tok.decode([token_id]).strip()
        rows.append({
            'rank': i + 1,
            'token': token,
            'token_id': token_id,
            'probability': float(top_probabilities[i]),
        })

    print(text)
    return pd.DataFrame(rows)


mask_predictions = predict_masked_token("Cambridge is the best university in [MASK]", top_k=5)
mask_predictions


Cambridge is the best university in [MASK]


,rank,token,token_id,probability
0,1,england,2563,0.392979
1,2,europe,2885,0.190914
2,3,britain,3725,0.075619
3,4,asia,4021,0.052152
4,5,.,1012,0.031965


### Checks (Exercise 3)

In [ ]:
assert isinstance(mask_predictions, pd.DataFrame)
required_columns = ['rank', 'token', 'token_id', 'probability']
for column in required_columns:
    assert column in mask_predictions.columns

assert len(mask_predictions) == 5
assert mask_predictions['probability'].between(0, 1).all()

probabilities = mask_predictions['probability'].tolist()
for i in range(len(probabilities) - 1):
    assert probabilities[i] >= probabilities[i + 1]

print('Exercise 3 passed.')


## 4. Exercise 4: GPT Next Tokens and Temperature

GPT-style causal language models predict the next token from left context only.

Task A:
- implement `predict_next_tokens(prompt, top_k=5)`,
- inspect the most likely next tokens after a prompt.

Task B:
- use `generate_text(prompt, temperature)` to compare low and high temperature generation.

Function contract for `predict_next_tokens`:
- input: one prompt string,
- output columns: `rank`, `token`, `token_id`, `probability`,
- predictions should come from the final position in the prompt.

Useful generation arguments:
- `max_new_tokens` controls length,
- `do_sample=True` enables sampling,
- lower `temperature` is more conservative,
- higher `temperature` is more diverse but less stable.


In [37]:
prompt = 'Deep learning is useful because'


def predict_next_tokens(prompt, top_k=5):
    rows = []

    inputs = gpt_tok(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = gpt_lm(**inputs)
    logits = outputs.logits[0, -1]

    probabilities = F.softmax(logits, dim=-1)
    top_probabilities, top_token_ids = torch.topk(probabilities, k=top_k)

    rows = []
    for i in range(top_k):
        token_id = int(top_token_ids[i])
        token = gpt_tok.decode([token_id]).strip()
        rows.append({
            'rank': i + 1,
            'token': token,
            'token_id': token_id,
            'probability': float(top_probabilities[i]),
        })
    # Store top_k next-token candidates.

    return pd.DataFrame(rows)


def generate_text(prompt, temperature=0.8, max_new_tokens=40):
    inputs = gpt_tok(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = gpt_lm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=gpt_tok.eos_token_id,
        )

    text = gpt_tok.decode(output_ids[0], skip_special_tokens=True)
    return text


next_token_predictions = predict_next_tokens(prompt, top_k=5)
low_temperature_text = generate_text(prompt, temperature=0.05, max_new_tokens=40)
no_temperature_text = generate_text(prompt, temperature=1.0, max_new_tokens=40)
high_temperature_text = generate_text(prompt, temperature=5000.0, max_new_tokens=40)

print(low_temperature_text)
print(no_temperature_text)
print(high_temperature_text)

next_token_predictions


Deep learning is useful because it allows us to learn from the past and to learn from the future.

























Deep learning is useful because it could be an even more important thing for people who are working on more complex learning. In addition, it can be a valuable way of learning and teaching people from a multitude of backgrounds to understand a
Deep learning is useful because they include both theoretical techniques at once of its potential! What the difference? First, theoretical physics could make data models simpler yet have huge theoretical dimensions of understanding them even faster and are much healthier or cheaper


,rank,token,token_id,probability
0,1,it,340,0.458646
1,2,we,356,0.104924
2,3,the,262,0.055949
3,4,you,345,0.049704
4,5,of,286,0.035837


### Checks (Exercise 4)

In [ ]:
assert isinstance(next_token_predictions, pd.DataFrame)
required_columns = ['rank', 'token', 'token_id', 'probability']
for column in required_columns:
    assert column in next_token_predictions.columns

assert len(next_token_predictions) == 5
assert next_token_predictions['probability'].between(0, 1).all()

probabilities = next_token_predictions['probability'].tolist()
for i in range(len(probabilities) - 1):
    assert probabilities[i] >= probabilities[i + 1]

assert isinstance(low_temperature_text, str)
assert isinstance(high_temperature_text, str)
assert len(low_temperature_text) > len(prompt)
assert len(high_temperature_text) > len(prompt)

print('Exercise 4 passed.')


## 5. Exercise 5: Encoder vs Decoder Evidence Table

Now summarize what you observed, not just what the lecture said.

Task:
- build a small comparison table for DistilBERT and DistilGPT2,
- base the rows on experiments above.

Required columns:
- `property`, `DistilBERT`, `DistilGPT2`.

Suggested rows:
- model family,
- context direction,
- tokenization observation,
- special tokens,
- masked-token prediction,
- next-token generation,
- typical use.


In [ ]:
comparison_rows = []

# Fill comparison_rows with evidence-based observations.
# Each row should be a dictionary with keys:
# property, DistilBERT, DistilGPT2

encoder_decoder_table = pd.DataFrame(comparison_rows)
encoder_decoder_table


### Checks (Exercise 5)

In [ ]:
assert isinstance(encoder_decoder_table, pd.DataFrame)
required_columns = ['property', 'DistilBERT', 'DistilGPT2']
for column in required_columns:
    assert column in encoder_decoder_table.columns

assert len(encoder_decoder_table) >= 5

for column in required_columns:
    assert encoder_decoder_table[column].isna().sum() == 0

print('Exercise 5 passed.')


## 6. Wrap-Up Questions

1. Why are subword tokenizers useful for rare or strange words?
2. What did BERT and GPT tokenizers represent differently?
3. What does `attention_mask = 0` mean in a padded batch?
4. Why can BERT use both left and right context for `[MASK]`?
5. Why is GPT generation sensitive to temperature?
6. When would you choose an encoder-style model, and when would you choose a decoder-style model?
